# Data Pre-processing

In [2]:
import sklearn as sk
import pandas as pd

## Import Data

In [3]:
df = pd.read_csv("US_Accidents_March23.csv")
len(df)

7728394

## Basic Data Preprocessing
To simplify your project a bit, use only the data instances that contain at most 1 missing value. According to our counts:
There are 3,554,549 data instances with no missing values:
There are 244,702 data instances with exactly one missing value.
Hence, your dataset for the project should contain 3,554,549 + 244,702 = 3,799,251

In [4]:
df.dropna(thresh=len(df.columns) - 1, inplace=True)
len(df)

3799251

In [5]:
df.drop(columns=['ID', 'Description', 'Country', 'Source', 'Turning_Loop'], inplace = True)
print(df.columns)
basic_df = df.copy(deep=True)

Index(['Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng',
       'End_Lat', 'End_Lng', 'Distance(mi)', 'Street', 'City', 'County',
       'State', 'Zipcode', 'Timezone', 'Airport_Code', 'Weather_Timestamp',
       'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
       'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
       'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing',
       'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station',
       'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset',
       'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight'],
      dtype='str')


## Advanced Pre-processing
Goals: 
- Encode all features to numeric values
- Use feature selection/elimination to reduce dimensionality for later analysis

In [6]:
advanced_df = df.copy(deep=True)

# Handle datetime -> integer by extracting integer values
advanced_df["Start_Time"] = pd.to_datetime(advanced_df["Start_Time"], format="ISO8601")
advanced_df["End_Time"] = pd.to_datetime(advanced_df["End_Time"], format="ISO8601")

advanced_df["Duration_Minutes"] = (
    advanced_df["End_Time"] - advanced_df["Start_Time"]
).dt.total_seconds() / 60.0
advanced_df["Start_Hour"] = advanced_df["Start_Time"].dt.hour
advanced_df["Start_DayOfWeek"] = advanced_df["Start_Time"].dt.dayofweek  # 0 = Monday, 6 = Sunday
advanced_df["Start_Month"] = advanced_df["Start_Time"].dt.month
advanced_df.drop(columns=["Start_Time", "End_Time"], inplace=True)


In [7]:
# Drop location cols other than lat/long
advanced_df.drop(
    columns=[
        "City", "County", "Zipcode", "Street", "Airport_Code", "State"
    ],
    inplace=True,
)

In [8]:
#Convert sunset measures
sunset_fields = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
for f in sunset_fields:
    advanced_df[f] = advanced_df[f].map({"Day": 1, "Night": 0}) # day = 1, night = 0

In [9]:
# One-hot encode wind direction and weather condition as dense columns
import numpy as np
from sklearn.preprocessing import OneHotEncoder

weather_columns = ["Wind_Direction", "Weather_Condition"]
encoder = OneHotEncoder(
    handle_unknown="ignore"
)

encoded_array = encoder.fit_transform(advanced_df[weather_columns]).toarray()
encoded_df = pd.DataFrame(
    encoded_array,
    index=advanced_df.index,
    columns=encoder.get_feature_names_out(),
)
advanced_df = advanced_df.drop(columns=weather_columns).join(encoded_df)


In [10]:
#Drop timezone, weather
advanced_df.drop(columns = ["Timezone", "Weather_Timestamp"], inplace = True)

In [11]:
advanced_df.info()

<class 'pandas.DataFrame'>
Index: 3799251 entries, 3402762 to 7728393
Columns: 187 entries, Severity to Weather_Condition_nan
dtypes: bool(12), float64(167), int32(3), int64(5)
memory usage: 5.0 GB


In [12]:
advanced_df.head()

,Severity,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),...,Weather_Condition_Thunder and Hail / Windy,Weather_Condition_Thunder in the Vicinity,Weather_Condition_Thunderstorm,Weather_Condition_Thunderstorms and Rain,Weather_Condition_Tornado,Weather_Condition_Widespread Dust,Weather_Condition_Widespread Dust / Windy,Weather_Condition_Wintry Mix,Weather_Condition_Wintry Mix / Windy,Weather_Condition_nan
3402762,3,40.108910,-83.092860,40.112060,-83.031870,3.230,42.1,36.1,58.0,29.76,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3402767,3,39.172393,-84.492792,39.170476,-84.501798,0.500,37.0,29.8,93.0,29.69,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3402768,2,39.063240,-84.032430,39.067310,-84.058510,1.427,35.6,29.2,100.0,29.66,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3402769,2,39.067080,-84.058550,39.063020,-84.032540,1.423,35.6,29.2,100.0,29.66,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3402771,2,41.375310,-81.820170,41.367860,-81.821740,0.521,33.1,30.0,92.0,29.63,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
basic_df.head()

,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Street,City,...,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
3402762,3,2016-02-08 00:37:08,2016-02-08 06:37:08,40.108910,-83.092860,40.112060,-83.031870,3.230,Outerbelt E,Dublin,...,False,False,False,False,False,False,Night,Night,Night,Night
3402767,3,2016-02-08 07:53:43,2016-02-08 13:53:43,39.172393,-84.492792,39.170476,-84.501798,0.500,I-75 S,Cincinnati,...,False,False,False,False,False,False,Day,Day,Day,Day
3402768,2,2016-02-08 08:16:57,2016-02-08 14:16:57,39.063240,-84.032430,39.067310,-84.058510,1.427,State Route 32,Williamsburg,...,False,False,False,False,False,True,Day,Day,Day,Day
3402769,2,2016-02-08 08:16:57,2016-02-08 14:16:57,39.067080,-84.058550,39.063020,-84.032540,1.423,State Route 32,Batavia,...,False,False,False,False,False,False,Day,Day,Day,Day
3402771,2,2016-02-08 11:51:46,2016-02-08 17:51:46,41.375310,-81.820170,41.367860,-81.821740,0.521,I-71 S,Cleveland,...,False,False,False,False,False,False,Day,Day,Day,Day


# Export

In [14]:
%pip install pyarrow

basic_df.to_parquet(
    "basic_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)

advanced_df.to_parquet(
    "advanced_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)

Note: you may need to restart the kernel to use updated packages.


# Classification

## Dummy Classifier

In [ ]:
Train = sk.model_selection.train_test_split(basic_df)[0]
print("Training data: ", Train)
imputer = sk.impute.SimpleImputer(strategy='mean')
imputer.fit(Train)

dummy = sk.dummy.DummyClassifier(strategy="most_frequent")
dummy_basic = Train.copy().select_dtypes(include=['number'])
dummy_basic.head()
severity = dummy_basic['Severity'].to_numpy()
dummy.fit(dummy_basic,severity)
dummy.predict(dummy_basic)


basicer = Train.copy().select_dtypes(include=['number'])
sk.metrics.confusion_matrix(basicer, dummy_basic)

print("Majority Class Classifier Score: ", dummy.score(dummy_basic, severity))

X           Severity                     Start_Time  \
4063208         2            2022-09-30 08:28:00   
7175015         2            2020-01-15 15:58:00   
6447237         2            2021-02-26 18:19:00   
4322108         4            2022-08-30 04:25:00   
4534682         2            2022-02-01 07:04:41   
...           ...                            ...   
6920526         2            2020-11-13 16:55:00   
3668954         2            2023-02-27 17:39:00   
5320429         2  2022-10-04 05:43:00.000000000   
5133579         2            2022-08-05 21:26:00   
6075566         2            2021-08-24 16:42:39   

                              End_Time  Start_Lat   Start_Lng    End_Lat  \
4063208            2022-09-30 08:53:00  38.909502  -77.005988  38.908538   
7175015            2020-01-15 16:50:22  34.083533 -117.544627  34.083533   
6447237            2021-02-26 20:00:00  36.098379  -78.734039  36.096981   
4322108            2022-08-30 06:26:31  38.000564  -79.173560  38.01

ValueError: Cannot use mean strategy with non-numeric data:
could not convert string to float: '2021-06-07 18:37:38'

## Decision Tree Classifier

## Random Forest Classifier

## 10 Fold Stratified Cross Verification

## Final Classification Model